In [1]:
from pathlib import Path

import polars as pl
import numpy as np
import scipy as sp

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dqdvs import dqdv_histogram

from config import DATA_PATH
ZENODO_DATA_PATH = Path(DATA_PATH) / Path("HALF_CELL_OCVS_ZENODO")

In [2]:
################################ LOAD DATA  ###################################
all_filepaths = ZENODO_DATA_PATH.rglob("*.parquet")

In [3]:
lfp_filename =  'sintef__sintef-lfp-R2032-gelon-fbeeaf__20250607__p-ocvhold__RT.bdf.parquet'

def generate_results_plot():
    
    fig = make_subplots(cols=2, shared_yaxes=True, horizontal_spacing=0.01, column_widths=[0.7, 0.3])
    fig.update_xaxes(title_text="Normalized Cumulative Capacity / 1", row=1, col=1)
    fig.update_xaxes(title_text="dq/dV / V<sup>-1</sup>", row=1, col=2)
    fig.update_yaxes(title_text="Voltage / V", row=1, col=1)

    return fig

colors = px.colors.qualitative.Plotly


## LFP Finite differences

In [4]:
df = pl.read_parquet(ZENODO_DATA_PATH / Path(lfp_filename))

df_subset = (df
             .filter(pl.col('Step Index / 1')==2)
             .filter(pl.col('Cycle Count / 1')==3)
             )

q = df_subset['Cumulative Capacity / Ah'].to_numpy()
v = df_subset['Voltage / V'].to_numpy()

q = q/np.max(q)
v_smooth = sp.ndimage.gaussian_filter1d(v, sigma=100) #100

v_bin_mon = np.maximum.accumulate(v_smooth)

mask = np.concatenate((np.array(v_bin_mon[:-1] != v_bin_mon[1:]), [True])) #flags duplicate voltage values in monotonic array, except the last of each duplicate.
v_bin_nodup = v_bin_mon[mask]
q_bin_nodup = q[mask]

dqdv = np.diff(q)/np.diff(v_smooth)
dqdv = np.append(dqdv, dqdv[-1])

dqdv_mon = np.diff(q_bin_nodup)/np.diff(v_bin_nodup)
dqdv_mon = np.append(dqdv_mon, dqdv_mon[-1])

fig = generate_results_plot()
fig.add_trace(go.Scatter(x=q, 
                         y=v, 
                         name="OCV", 
                         mode="lines",
                         line=dict(color="Black", width=4)), row=1, col=1)

fig.add_trace(go.Scatter(x=q, 
                         y=v_smooth, 
                         name="OCV Smooth", 
                         mode="lines",
                         line=dict(color=colors[0], width=2)), row=1, col=1)

fig.add_trace(go.Scatter(x=q_bin_nodup, 
                         y=v_bin_nodup, 
                         name="OCV Monotonic", 
                         mode="lines",
                         line=dict(color=colors[1], width=2, dash="dot" )), row=1, col=1)

fig.add_trace(go.Scatter(x=dqdv, 
                         y=v_smooth, 
                         name="dq/dV smoothing", 
                         mode="lines",
                         line=dict(color=colors[0], width=2)), row=1, col=2)

fig.add_trace(go.Scatter(x=dqdv_mon, 
                         y=v_bin_mon, 
                         name="dq/dV Monotonic", 
                         mode="lines",
                         line=dict(color=colors[1], width=2, dash="dot" )), row=1, col=2)

fig.add_annotation(x=0.04, y=3.4453,
            text="Maximum",
            showarrow=True,
            arrowhead=0, ax=30, ay=-30)

fig.add_annotation(x=0.14, y=3.4421,
            text="Minimum",
            showarrow=True,
            arrowhead=0, ax=8, ay=30)

fig.update_layout(
        template="ggplot2",
        margin=dict(l=10, r=10, t=70, b=10),
        width=600, 
        height=350,
        legend=dict(
            orientation="h",    # horizontal
            yanchor="bottom",   # anchor legend's bottom
            y=1.0,             # place above the plot area
            xanchor="center",
            x=0.5
        ),)



fig.update_yaxes(showgrid=True, ticks="")
fig.update_yaxes(range=[3.43, 3.457])

fig.update_xaxes(showgrid=False)
fig.update_xaxes(range=[0, 0.99], row=1, col=1)
fig.update_xaxes(range=[-1e5, 5.99e5], row=1, col=2)

fig.update_layout(title=dict(text="LiFePO<sub>4</sub> vs Li: finite differences", 
                            yanchor="bottom",   # anchor legend's bottom
                            y=0.95,             # place above the plot area
                            xanchor="left",
                            x=0.05))



In [5]:
fig.write_image("figures/lfp_finite_differences.png", scale=5)

## LFP with histogram method

In [6]:
BIN_SIZE = 1e-3
v_dqdv, dqdv = dqdv_histogram(q=q, v=v, bin_size=BIN_SIZE, smooth=True)

q_reconstructed = sp.integrate.cumulative_trapezoid(dqdv, v_dqdv, initial=0.0) + q[0]

fig = generate_results_plot()
fig.add_trace( go.Scatter(x=q, 
                          y=v, 
                          name="OCV", 
                          line=dict(color="Black", width=4)), 
                        row=1, col=1 )
fig.add_trace( go.Scatter(x=q_reconstructed, 
                          y=v_dqdv, 
                          name="OCV reconstructed", 
                          line=dict(color=colors[0], width=3, dash="dot")), 
                        row=1, col=1 )
fig.add_trace( go.Scatter(x=dqdv, 
                          y=v_dqdv, 
                          name="OCV histogram: 1 mV bin size", 
                          line=dict(color=colors[0], width=2)), 
                        row=1, col=2 )
fig.add_trace(go.Bar(x=dqdv, 
                     y=v_dqdv, 
                     name="Bins", 
                     orientation="h", 
                     opacity=0.5, 
                     marker=dict(color=colors[0]), 
                     showlegend=False), 
                    row=1, col=2 )

fig.update_layout(
        template="ggplot2",
        margin=dict(l=10, r=10, t=70, b=10),
        width=600, 
        height=350,
        legend=dict(
            orientation="h",    # horizontal
            yanchor="bottom",   # anchor legend's bottom
            y=1.0,             # place above the plot area
            xanchor="center",
            x=0.5
        ),)

fig.update_yaxes(range=[3.43, 3.459], showgrid=True, ticks="")
fig.update_xaxes(showgrid=False)
fig.update_xaxes(range=[0.01, 0.98], row=1, col=1)
fig.update_layout(title=dict(text="LiFePO<sub>4</sub> vs Li: finite differences", 
                            yanchor="bottom",   # anchor legend's bottom
                            y=0.93,             # place above the plot area
                            xanchor="left",
                            x=0.05))
fig.show()

In [7]:
fig.write_image("figures/lfp_histogram.png", scale=5)